# Brain Metastasis Ensemble Fine-tuning

Fine-tune pretrained models on labeled segmentation data.

**Prerequisites:** Run `pretrain_ensemble_colab.ipynb` first to generate pretrained weights.

**Features:**
- Loads pretrained weights from masked reconstruction
- Fine-tunes on labeled segmentation data with Tversky loss (high sensitivity)
- Fully resumable - safe to stop/restart anytime
- Tracks Dice, sensitivity, specificity metrics
- Progress saved to Drive after each epoch

**Setup:**
1. Run cells 1-5 in order (setup)
2. Run cell 6 to start/resume fine-tuning

**If session disconnects:** Run cells 1-5, then cell 6 - it will auto-resume.

In [ ]:
# 1. Check GPU & Mount Drive
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/brainMetShare'
DRIVE_DIR = '/content/drive/MyDrive/BrainMetShare'
LOCAL_DATA = '/content/data'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/model/training_states", exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print("\nDirectories created!")

In [ ]:
# 2. Install Dependencies
!pip install nibabel scipy tqdm monai -q
print("Dependencies installed!")

In [ ]:
# 3. Upload Code
from google.colab import files
import zipfile

print("Upload brainMetShare_code.zip")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as z:
            z.extractall('/content/brainMetShare')
        print(f"Extracted {filename}")

# Verify
src_path = '/content/brainMetShare/src/segmentation'
if os.path.exists(src_path):
    print(f"Code ready: {os.listdir(src_path)}")
else:
    print("WARNING: src/segmentation not found - check zip structure")

In [ ]:
# 4. Copy Training Data to Local SSD
import time
import shutil

start = time.time()
!df -h /content

# Create local data directory
os.makedirs(LOCAL_DATA, exist_ok=True)

# Check available data sources (need LABELED data with masks for fine-tuning)
SUPERSET_DIR = f"{DRIVE_DIR}/Superset"
TRAIN_SRC = f"{SUPERSET_DIR}/full/train"
ALT_TRAIN_SRC = f"{DRIVE_DIR}/preprocessed_256/train"

print("Checking data sources...")
print(f"  Superset train: {os.path.exists(TRAIN_SRC)}")
print(f"  Alt train (preprocessed_256): {os.path.exists(ALT_TRAIN_SRC)}")

# Copy train data
LOCAL_TRAIN = f"{LOCAL_DATA}/train"
train_src = TRAIN_SRC if os.path.exists(TRAIN_SRC) else ALT_TRAIN_SRC

if os.path.exists(train_src):
    print(f"\nCopying train data from {train_src}...")
    !cp -r "{train_src}" "{LOCAL_TRAIN}"
    n_train = len([d for d in os.listdir(LOCAL_TRAIN) if os.path.isdir(f"{LOCAL_TRAIN}/{d}")])
    print(f"Train: {n_train} cases")
else:
    print("ERROR: No train data found!")

print(f"\nData copied in {(time.time()-start)/60:.1f} min")
!df -h /content

In [ ]:
# 5. Setup Python Path & Restore Previous State
import sys
sys.path.insert(0, '/content/brainMetShare/src')
os.chdir(PROJECT_DIR)

# Restore training states from Drive
DRIVE_STATES = f"{DRIVE_DIR}/model/training_states"
LOCAL_STATES = f"{PROJECT_DIR}/model/training_states"

if os.path.exists(DRIVE_STATES):
    print("Restoring training states from Drive...")
    for f in os.listdir(DRIVE_STATES):
        src = f"{DRIVE_STATES}/{f}"
        dst = f"{LOCAL_STATES}/{f}"
        if os.path.isfile(src):
            shutil.copy(src, dst)
            print(f"  Restored: {f}")

# Restore pretrained and finetuned models from Drive
DRIVE_MODELS = f"{DRIVE_DIR}/model"
LOCAL_MODELS = f"{PROJECT_DIR}/model"

if os.path.exists(DRIVE_MODELS):
    print("\nRestoring model checkpoints from Drive...")
    for f in os.listdir(DRIVE_MODELS):
        if f.endswith('.pth'):
            src = f"{DRIVE_MODELS}/{f}"
            dst = f"{LOCAL_MODELS}/{f}"
            if os.path.isfile(src):
                shutil.copy(src, dst)
                print(f"  Restored: {f}")

# Check pretrained models exist
print("\nPretrained models available:")
for name in ['exp1_8patch', 'exp3_12patch_maxfn', 'improved_24patch', 'improved_36patch']:
    path = f"{LOCAL_MODELS}/{name}_pretrained.pth"
    status = "YES" if os.path.exists(path) else "NO"
    print(f"  {name}: {status}")

print("\nReady for fine-tuning!")

In [ ]:
# 6. Fine-tune Ensemble (Resumable)
import gc
import json
import shutil
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast, GradScaler
import numpy as np
import nibabel as nib
from tqdm import tqdm

from segmentation.unet import LightweightUNet3D
from segmentation.dataset import BrainMetDataset

# =============================================================================
# CONFIGURATION
# =============================================================================
ENSEMBLE_CONFIGS = {
    'exp1_8patch': {
        'patch_size': 8,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'finetune_epochs': 50,
        'optimal_threshold': 0.3,
    },
    'exp3_12patch_maxfn': {
        'patch_size': 12,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'finetune_epochs': 50,
        'optimal_threshold': 0.25,
    },
    'improved_24patch': {
        'patch_size': 24,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'finetune_epochs': 50,
        'optimal_threshold': 0.5,
    },
    'improved_36patch': {
        'patch_size': 36,
        'base_channels': 20,
        'use_attention': True,
        'use_residual': True,
        'finetune_epochs': 50,
        'optimal_threshold': 0.5,
    },
}

# Try t2 first (Superset), fall back to bravo (BrainMetShare)
SEQUENCES = ['t1_pre', 't1_gd', 'flair', 't2']
ALT_SEQUENCES = ['t1_pre', 't1_gd', 'flair', 'bravo']

BATCH_SIZE = 2  # Smaller batch for fine-tuning (more memory for gradients)
LR = 0.0003  # Lower LR for fine-tuning

# Paths
TRAIN_DATA = LOCAL_TRAIN
MODEL_DIR = Path(f"{PROJECT_DIR}/model")
STATE_DIR = MODEL_DIR / "training_states"
DRIVE_MODEL_DIR = Path(f"{DRIVE_DIR}/model")
DRIVE_STATE_DIR = DRIVE_MODEL_DIR / "training_states"

DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_STATE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# =============================================================================
# TVERSKY LOSS (High Sensitivity)
# =============================================================================
class TverskyLoss(nn.Module):
    """Tversky loss with alpha < beta penalizes false negatives more."""
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha = alpha  # FP weight
        self.beta = beta    # FN weight (higher = penalize missed lesions more)
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        tp = (pred * target).sum()
        fp = (pred * (1 - target)).sum()
        fn = ((1 - pred) * target).sum()
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return 1 - tversky

# =============================================================================
# COMBINED LOSS
# =============================================================================
class CombinedLoss(nn.Module):
    """Combined Tversky + Focal loss for better small lesion detection."""
    def __init__(self):
        super().__init__()
        self.tversky = TverskyLoss(alpha=0.3, beta=0.7)

    def focal_loss(self, pred, target, alpha=0.75, gamma=2.0):
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        return (alpha * (1 - pt) ** gamma * bce).mean()

    def forward(self, pred, target):
        return 0.7 * self.tversky(pred, target) + 0.3 * self.focal_loss(pred, target)

# =============================================================================
# METRICS
# =============================================================================
def compute_metrics(pred, target, threshold=0.5):
    pred_binary = (torch.sigmoid(pred) > threshold).float()

    tp = ((pred_binary == 1) & (target == 1)).sum().float()
    tn = ((pred_binary == 0) & (target == 0)).sum().float()
    fp = ((pred_binary == 1) & (target == 0)).sum().float()
    fn = ((pred_binary == 0) & (target == 1)).sum().float()

    dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
    sensitivity = (tp + 1e-6) / (tp + fn + 1e-6)
    specificity = (tn + 1e-6) / (tn + fp + 1e-6)
    precision = (tp + 1e-6) / (tp + fp + 1e-6)

    return {
        'dice': dice.item(),
        'sensitivity': sensitivity.item(),
        'specificity': specificity.item(),
        'precision': precision.item(),
    }

# =============================================================================
# TRAINING STATE MANAGER
# =============================================================================
class StateManager:
    def __init__(self, local_dir, drive_dir):
        self.local_dir = Path(local_dir)
        self.drive_dir = Path(drive_dir)

    def get_path(self, model_name):
        return self.local_dir / f'{model_name}_finetune_state.pth'

    def save(self, model_name, epoch, model, optimizer, scheduler, scaler, best_dice, history):
        state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_dice': best_dice,
            'history': history,
            'timestamp': datetime.now().isoformat()
        }

        local_path = self.get_path(model_name)
        torch.save(state, local_path)

        # Also save to Drive
        drive_path = self.drive_dir / f'{model_name}_finetune_state.pth'
        shutil.copy(local_path, drive_path)

    def load(self, model_name, model, optimizer, scheduler, scaler, device):
        local_path = self.get_path(model_name)
        if not local_path.exists():
            return None

        print(f"Loading state from {local_path}")
        state = torch.load(local_path, map_location=device, weights_only=False)
        model.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        scheduler.load_state_dict(state['scheduler_state_dict'])
        scaler.load_state_dict(state['scaler_state_dict'])
        print(f"Resumed from epoch {state['epoch']}, best_dice {state['best_dice']:.4f}")
        return state

    def is_complete(self, model_name):
        return (self.local_dir / f'{model_name}_finetune_DONE').exists()

    def mark_complete(self, model_name):
        (self.local_dir / f'{model_name}_finetune_DONE').touch()
        (self.drive_dir / f'{model_name}_finetune_DONE').touch()

state_manager = StateManager(STATE_DIR, DRIVE_STATE_DIR)

# =============================================================================
# FINE-TUNE ONE MODEL
# =============================================================================
def finetune_model(model_name, config):
    patch_size = config['patch_size']
    epochs = config['finetune_epochs']
    threshold = config['optimal_threshold']

    print(f"\n{'='*60}")
    print(f"FINE-TUNING: {model_name} (patch_size={patch_size})")
    print(f"{'='*60}")

    # Check if already complete
    if state_manager.is_complete(model_name):
        print(f"Already complete - skipping")
        return MODEL_DIR / f'{model_name}_finetuned.pth'

    # Create model
    model = LightweightUNet3D(
        in_channels=4,
        out_channels=1,
        base_channels=config['base_channels'],
        use_attention=config['use_attention'],
        use_residual=config['use_residual']
    ).to(device)

    # Load pretrained weights if available
    pretrained_path = MODEL_DIR / f'{model_name}_pretrained.pth'
    if pretrained_path.exists():
        print(f"Loading pretrained weights from {pretrained_path}")
        checkpoint = torch.load(pretrained_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"  Pretrain loss was: {checkpoint.get('loss', 'N/A')}")
    else:
        print(f"WARNING: No pretrained weights found at {pretrained_path}")
        print(f"  Training from scratch...")

    # Dataset
    try:
        dataset = BrainMetDataset(
            data_dir=str(TRAIN_DATA),
            sequences=SEQUENCES,
            patch_size=(patch_size, patch_size, patch_size),
            target_size=(128, 128, 128)
        )
    except:
        print(f"  Falling back to alternate sequences: {ALT_SEQUENCES}")
        dataset = BrainMetDataset(
            data_dir=str(TRAIN_DATA),
            sequences=ALT_SEQUENCES,
            patch_size=(patch_size, patch_size, patch_size),
            target_size=(128, 128, 128)
        )

    # Train/val split (85/15)
    n_val = int(len(dataset) * 0.15)
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

    print(f"Train: {n_train} cases, Val: {n_val} cases")
    print(f"Batches/epoch: {len(train_loader)}")

    # Loss
    criterion = CombinedLoss()

    # Optimizer (lower LR for fine-tuning)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler('cuda')

    # Try to resume
    start_epoch = 1
    best_dice = 0
    history = {'train_loss': [], 'val_dice': [], 'val_sens': [], 'val_spec': [], 'lr': []}

    state = state_manager.load(model_name, model, optimizer, scheduler, scaler, device)
    if state:
        start_epoch = state['epoch'] + 1
        best_dice = state['best_dice']
        history = state.get('history', history)

        if start_epoch > epochs:
            print(f"Training already complete")
            state_manager.mark_complete(model_name)
            return MODEL_DIR / f'{model_name}_finetuned.pth'

    checkpoint_path = MODEL_DIR / f'{model_name}_finetuned.pth'

    # Training loop
    for epoch in range(start_epoch, epochs + 1):
        # Train
        model.train()
        train_loss = 0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"Train {epoch}/{epochs}")
        for images, masks, _ in pbar:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()

            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        # Validate
        model.eval()
        val_metrics = {'dice': 0, 'sensitivity': 0, 'specificity': 0}
        n_val_batches = 0

        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.to(device), masks.to(device)

                with autocast('cuda'):
                    outputs = model(images)

                metrics = compute_metrics(outputs, masks, threshold=threshold)
                for k in val_metrics:
                    val_metrics[k] += metrics[k]
                n_val_batches += 1

        scheduler.step()

        # Average metrics
        avg_train_loss = train_loss / n_batches
        for k in val_metrics:
            val_metrics[k] /= n_val_batches
        current_lr = scheduler.get_last_lr()[0]

        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_dice'].append(val_metrics['dice'])
        history['val_sens'].append(val_metrics['sensitivity'])
        history['val_spec'].append(val_metrics['specificity'])
        history['lr'].append(current_lr)

        print(f"Epoch {epoch}: Loss={avg_train_loss:.4f}, Dice={val_metrics['dice']:.4f}, "
              f"Sens={val_metrics['sensitivity']:.4f}, Spec={val_metrics['specificity']:.4f}")

        # Save best
        if val_metrics['dice'] > best_dice:
            best_dice = val_metrics['dice']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'dice': best_dice,
                'sensitivity': val_metrics['sensitivity'],
                'specificity': val_metrics['specificity'],
                'config': config
            }, checkpoint_path)
            # Also save to Drive
            shutil.copy(checkpoint_path, DRIVE_MODEL_DIR / f'{model_name}_finetuned.pth')
            print(f"  Saved best model (Dice={best_dice:.4f})")

        # Save state (for resume)
        state_manager.save(model_name, epoch, model, optimizer, scheduler, scaler, best_dice, history)
        print(f"  [Checkpoint saved - safe to stop]")

        if epoch % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    state_manager.mark_complete(model_name)
    print(f"\nFine-tuning complete! Best Dice: {best_dice:.4f}")
    return checkpoint_path

# =============================================================================
# RUN FINE-TUNING
# =============================================================================
print("\n" + "="*60)
print("ENSEMBLE FINE-TUNING")
print("="*60)
print(f"\nModels to fine-tune: {list(ENSEMBLE_CONFIGS.keys())}")
print(f"Data directory: {TRAIN_DATA}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LR}")

results = {}
for model_name, config in ENSEMBLE_CONFIGS.items():
    checkpoint = finetune_model(model_name, config)
    results[model_name] = str(checkpoint) if checkpoint else None

    gc.collect()
    torch.cuda.empty_cache()

# Summary
print("\n" + "="*60)
print("FINE-TUNING COMPLETE")
print("="*60)
for model_name, path in results.items():
    status = "Done" if path else "Failed"
    print(f"  {model_name}: {status}")
print(f"\nModels saved to: {DRIVE_MODEL_DIR}")

In [ ]:
# 6b. Full-Volume Validation (Run after training completes)
# This gives accurate metrics by running inference on complete 3D volumes

import torch
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from pathlib import Path
from tqdm import tqdm
import json

from segmentation.unet import LightweightUNet3D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =============================================================================
# SLIDING WINDOW INFERENCE
# =============================================================================
def sliding_window_inference(model, volume, patch_size, overlap=0.5, threshold=0.5):
    """
    Run inference on full volume using sliding window with overlap.
    
    Args:
        model: Trained model
        volume: Input volume (C, H, W, D)
        patch_size: Size of patches (int)
        overlap: Overlap fraction between patches
        threshold: Binarization threshold
    
    Returns:
        Binary prediction volume (H, W, D)
    """
    model.eval()
    C, H, W, D = volume.shape
    p = patch_size
    stride = int(p * (1 - overlap))
    
    # Pad volume if needed
    pad_h = (p - H % p) % p if H % stride != 0 else 0
    pad_w = (p - W % p) % p if W % stride != 0 else 0
    pad_d = (p - D % p) % p if D % stride != 0 else 0
    
    if pad_h > 0 or pad_w > 0 or pad_d > 0:
        volume = F.pad(torch.from_numpy(volume), (0, pad_d, 0, pad_w, 0, pad_h), mode='constant', value=0).numpy()
        C, H, W, D = volume.shape
    
    # Initialize output and count arrays
    output = np.zeros((H, W, D), dtype=np.float32)
    count = np.zeros((H, W, D), dtype=np.float32)
    
    # Generate patch coordinates
    coords = []
    for h in range(0, H - p + 1, stride):
        for w in range(0, W - p + 1, stride):
            for d in range(0, D - p + 1, stride):
                coords.append((h, w, d))
    
    # Process patches in batches
    batch_size = 8
    with torch.no_grad():
        for i in range(0, len(coords), batch_size):
            batch_coords = coords[i:i+batch_size]
            patches = []
            
            for h, w, d in batch_coords:
                patch = volume[:, h:h+p, w:w+p, d:d+p]
                patches.append(patch)
            
            # Stack and predict
            batch = torch.from_numpy(np.stack(patches)).float().to(device)
            with torch.amp.autocast('cuda'):
                preds = torch.sigmoid(model(batch)).cpu().numpy()
            
            # Accumulate predictions
            for j, (h, w, d) in enumerate(batch_coords):
                output[h:h+p, w:w+p, d:d+p] += preds[j, 0]
                count[h:h+p, w:w+p, d:d+p] += 1
    
    # Average overlapping regions
    output = output / np.maximum(count, 1)
    
    # Remove padding
    if pad_h > 0 or pad_w > 0 or pad_d > 0:
        output = output[:H-pad_h if pad_h else H, :W-pad_w if pad_w else W, :D-pad_d if pad_d else D]
    
    # Binarize
    return (output > threshold).astype(np.float32), output

# =============================================================================
# METRICS ON FULL VOLUMES
# =============================================================================
def compute_volume_metrics(pred, target):
    """Compute metrics on full volume."""
    pred = pred.flatten()
    target = target.flatten()
    
    tp = ((pred == 1) & (target == 1)).sum()
    tn = ((pred == 0) & (target == 0)).sum()
    fp = ((pred == 1) & (target == 0)).sum()
    fn = ((pred == 0) & (target == 1)).sum()
    
    dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
    sensitivity = (tp + 1e-6) / (tp + fn + 1e-6)
    specificity = (tn + 1e-6) / (tn + fp + 1e-6)
    precision = (tp + 1e-6) / (tp + fp + 1e-6)
    
    return {
        'dice': float(dice),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'precision': float(precision),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn)
    }

# =============================================================================
# LOAD AND PREPROCESS VOLUME
# =============================================================================
def load_volume(case_dir, sequences, target_size=(128, 128, 128)):
    """Load and preprocess a full volume."""
    from scipy.ndimage import zoom
    
    images = []
    for seq in sequences:
        path = case_dir / f"{seq}.nii.gz"
        if path.exists():
            data = nib.load(str(path)).get_fdata().astype(np.float32)
            # Resize
            factors = [t / s for t, s in zip(target_size, data.shape)]
            data = zoom(data, factors, order=1)
            # Normalize
            mean, std = data.mean(), data.std()
            data = (data - mean) / (std + 1e-6)
            images.append(data)
        else:
            images.append(np.zeros(target_size, dtype=np.float32))
    
    return np.stack(images, axis=0)  # (C, H, W, D)

def load_mask(case_dir, target_size=(128, 128, 128)):
    """Load ground truth mask."""
    from scipy.ndimage import zoom
    
    # Try different mask names
    for mask_name in ['seg.nii.gz', 'mask.nii.gz', 'label.nii.gz']:
        path = case_dir / mask_name
        if path.exists():
            data = nib.load(str(path)).get_fdata().astype(np.float32)
            factors = [t / s for t, s in zip(target_size, data.shape)]
            data = zoom(data, factors, order=0)  # Nearest neighbor for masks
            return (data > 0.5).astype(np.float32)
    
    return None

# =============================================================================
# RUN FULL-VOLUME VALIDATION
# =============================================================================
print("="*60)
print("FULL-VOLUME VALIDATION")
print("="*60)

# Configuration
SEQUENCES = ['t1_pre', 't1_gd', 'flair', 't2']
ALT_SEQUENCES = ['t1_pre', 't1_gd', 'flair', 'bravo']
TARGET_SIZE = (128, 128, 128)

# Get validation cases (use same split as training - last 15%)
all_cases = sorted([d for d in Path(LOCAL_TRAIN).iterdir() if d.is_dir()])
n_val = int(len(all_cases) * 0.15)
val_cases = all_cases[-n_val:]  # Last 15% for validation

print(f"\nValidating on {len(val_cases)} full volumes")

# Check which sequences are available
test_case = val_cases[0]
if (test_case / 't2.nii.gz').exists():
    sequences = SEQUENCES
else:
    sequences = ALT_SEQUENCES
print(f"Using sequences: {sequences}")

# Models to validate
MODELS_TO_VALIDATE = {
    'exp1_8patch': {'patch_size': 8, 'threshold': 0.3},
    'exp3_12patch_maxfn': {'patch_size': 12, 'threshold': 0.25},
    'improved_24patch': {'patch_size': 24, 'threshold': 0.5},
    'improved_36patch': {'patch_size': 36, 'threshold': 0.5},
}

all_results = {}

for model_name, config in MODELS_TO_VALIDATE.items():
    print(f"\n{'='*60}")
    print(f"Validating: {model_name}")
    print(f"{'='*60}")
    
    # Load model
    model_path = MODEL_DIR / f'{model_name}_finetuned.pth'
    if not model_path.exists():
        print(f"  Model not found: {model_path}")
        continue
    
    model = LightweightUNet3D(
        in_channels=4, out_channels=1,
        base_channels=20, use_attention=True, use_residual=True
    ).to(device)
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    print(f"  Loaded model (train Dice: {checkpoint.get('dice', 'N/A')})")
    
    # Validate on each case
    case_metrics = []
    
    for case_dir in tqdm(val_cases, desc="  Processing"):
        # Load volume and mask
        volume = load_volume(case_dir, sequences, TARGET_SIZE)
        mask = load_mask(case_dir, TARGET_SIZE)
        
        if mask is None:
            continue
        
        # Run inference
        pred_binary, pred_prob = sliding_window_inference(
            model, volume,
            patch_size=config['patch_size'],
            overlap=0.5,
            threshold=config['threshold']
        )
        
        # Compute metrics
        metrics = compute_volume_metrics(pred_binary, mask)
        case_metrics.append(metrics)
    
    # Aggregate results
    if case_metrics:
        avg_metrics = {
            'dice': np.mean([m['dice'] for m in case_metrics]),
            'sensitivity': np.mean([m['sensitivity'] for m in case_metrics]),
            'specificity': np.mean([m['specificity'] for m in case_metrics]),
            'precision': np.mean([m['precision'] for m in case_metrics]),
            'std_dice': np.std([m['dice'] for m in case_metrics]),
            'n_cases': len(case_metrics)
        }
        
        all_results[model_name] = avg_metrics
        
        print(f"\n  Results ({len(case_metrics)} cases):")
        print(f"    Dice:        {avg_metrics['dice']:.4f} +/- {avg_metrics['std_dice']:.4f}")
        print(f"    Sensitivity: {avg_metrics['sensitivity']:.4f}")
        print(f"    Specificity: {avg_metrics['specificity']:.4f}")
        print(f"    Precision:   {avg_metrics['precision']:.4f}")

# Summary table
print("\n" + "="*60)
print("FULL-VOLUME VALIDATION SUMMARY")
print("="*60)
print(f"\n{'Model':<25} {'Dice':<12} {'Sens':<12} {'Spec':<12} {'Prec':<12}")
print("-"*65)

for model_name, metrics in all_results.items():
    print(f"{model_name:<25} {metrics['dice']:.4f}       {metrics['sensitivity']:.4f}       "
          f"{metrics['specificity']:.4f}       {metrics['precision']:.4f}")

# Save results
results_path = DRIVE_MODEL_DIR / 'full_volume_validation.json'
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to: {results_path}")

In [ ]:
# 7. Check Progress
import json
from pathlib import Path

print("Fine-tuning Status:")
print("="*60)

for model_name in ENSEMBLE_CONFIGS.keys():
    state_file = STATE_DIR / f'{model_name}_finetune_state.pth'
    done_file = STATE_DIR / f'{model_name}_finetune_DONE'

    if done_file.exists():
        print(f"{model_name}: COMPLETE")
    elif state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        epochs_total = ENSEMBLE_CONFIGS[model_name]['finetune_epochs']
        history = state.get('history', {})
        last_dice = history.get('val_dice', [0])[-1] if history.get('val_dice') else 0
        last_sens = history.get('val_sens', [0])[-1] if history.get('val_sens') else 0
        print(f"{model_name}: Epoch {state['epoch']}/{epochs_total}, "
              f"Best Dice={state['best_dice']:.4f}, Last Dice={last_dice:.4f}, Sens={last_sens:.4f}")
    else:
        print(f"{model_name}: Not started")

In [ ]:
# 8. Plot Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, model_name in enumerate(ENSEMBLE_CONFIGS.keys()):
    ax = axes[idx]
    state_file = STATE_DIR / f'{model_name}_finetune_state.pth'

    if state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        history = state.get('history', {})

        if history.get('val_dice') and len(history['val_dice']) > 0:
            epochs = range(1, len(history['val_dice']) + 1)
            ax.plot(epochs, history['val_dice'], 'b-', label='Dice', linewidth=2)
            ax.plot(epochs, history['val_sens'], 'g--', label='Sensitivity', linewidth=1.5)
            ax.plot(epochs, history['val_spec'], 'r:', label='Specificity', linewidth=1.5)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Score')
            ax.legend()
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            ax.set_title(f"{model_name} (best Dice={state['best_dice']:.4f})")
        else:
            ax.text(0.5, 0.5, 'No history', ha='center', va='center')
            ax.set_title(model_name)
    else:
        ax.text(0.5, 0.5, 'Not started', ha='center', va='center')
        ax.set_title(model_name)

plt.tight_layout()
plt.savefig(f"{DRIVE_MODEL_DIR}/finetune_curves.png", dpi=150)
plt.show()
print(f"Saved to {DRIVE_MODEL_DIR}/finetune_curves.png")

In [ ]:
# 9. Compare Pretrained vs Non-Pretrained (Optional)
# Run this after fine-tuning completes to see if pretraining helped

print("Model Comparison:")
print("="*70)
print(f"{'Model':<25} {'Finetuned Dice':<18} {'Original Best Dice':<18}")
print("-"*70)

# Original model results (from your leaderboard)
original_results = {
    'exp1_8patch': 0.78,  # Update with actual values
    'exp3_12patch_maxfn': 0.78,
    'improved_24patch': 0.88,
    'improved_36patch': 0.90,
}

for model_name in ENSEMBLE_CONFIGS.keys():
    # Get finetuned result
    state_file = STATE_DIR / f'{model_name}_finetune_state.pth'
    if state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        finetuned_dice = state['best_dice']
    else:
        finetuned_dice = 0

    original_dice = original_results.get(model_name, 0)
    diff = finetuned_dice - original_dice
    diff_str = f"({'+' if diff >= 0 else ''}{diff:.2%})"

    print(f"{model_name:<25} {finetuned_dice:<18.4f} {original_dice:<18.2f} {diff_str}")

In [ ]:
# 10. Final Save to Drive
import shutil
from datetime import datetime

print("Saving all models and states to Drive...")

# Save models
for f in MODEL_DIR.glob('*.pth'):
    dst = DRIVE_MODEL_DIR / f.name
    shutil.copy(f, dst)
    print(f"  Saved: {f.name}")

# Save states
for f in STATE_DIR.glob('*'):
    dst = DRIVE_STATE_DIR / f.name
    if f.is_file():
        shutil.copy(f, dst)

# Save summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'phase': 'finetune',
    'models': {}
}

for model_name in ENSEMBLE_CONFIGS.keys():
    state_file = STATE_DIR / f'{model_name}_finetune_state.pth'
    done_file = STATE_DIR / f'{model_name}_finetune_DONE'

    if state_file.exists():
        state = torch.load(state_file, map_location='cpu', weights_only=False)
        history = state.get('history', {})
        summary['models'][model_name] = {
            'epoch': state['epoch'],
            'best_dice': state['best_dice'],
            'final_sensitivity': history.get('val_sens', [0])[-1] if history.get('val_sens') else 0,
            'final_specificity': history.get('val_spec', [0])[-1] if history.get('val_spec') else 0,
            'complete': done_file.exists()
        }

with open(DRIVE_MODEL_DIR / 'finetune_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nAll files saved to: {DRIVE_MODEL_DIR}")
print("\nSummary:")
print(json.dumps(summary, indent=2))